# Polaris Support Tickets — EDA

Exploring a **~24k synthetic support-ticket dataset** with coherent triage labels
and a **temporal event layer** (outages + product launches, Jan 2024 – Jun 2026).
Full pipeline & modeling: https://github.com/VladislavMarinovich/saas-support-rag-triage

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
ACCENT, BASE_C, EVT_C = "#2E5A87", "#9db8d2", "#d9534f"

# load the dataset — Kaggle mounts it under /kaggle/input; fall back to local
parq = glob.glob("/kaggle/input/*/polaris_tickets_v2.parquet")
if parq:
    df = pd.read_parquet(parq[0])
elif os.path.exists("data/tickets_v2.jsonl"):
    df = pd.read_json("data/tickets_v2.jsonl", lines=True)
else:
    df = pd.read_parquet(glob.glob("**/polaris_tickets_v2.parquet", recursive=True)[0])

df["created_at"] = pd.to_datetime(df["created_at"])
df["month"] = df["created_at"].dt.to_period("M").dt.to_timestamp()
df["is_event"] = df["event_id"].notna()
print(df.shape); df.head(3)

## At a glance

In [ ]:
n = len(df); ev = int(df.is_event.sum())
print(f"tickets      : {n:,}")
print(f"span         : {df.created_at.min().date()} -> {df.created_at.max().date()}")
print(f"event-driven : {ev:,} ({ev/n*100:.1f}%)")
print(df.groupby(df.created_at.dt.year).size().rename("tickets").to_frame())

## 1. Temporal signature — outage spikes & launch waves

Monthly volume split into baseline flow and event-driven tickets.

In [ ]:
piv = (df.groupby(["month", "is_event"]).size()
         .unstack(fill_value=0).rename(columns={False: "baseline", True: "event"}))
piv.index = piv.index.strftime("%Y-%m")
ax = piv[["baseline", "event"]].plot(kind="bar", stacked=True, figsize=(14, 5),
                                     color=[BASE_C, EVT_C], width=0.9)
ax.set_title("Monthly ticket volume — baseline vs event-driven", fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("tickets")
ax.tick_params(axis="x", labelrotation=90, labelsize=7); ax.legend(title="")
plt.tight_layout(); plt.show()

## 2. The launch arc — requests that vanish on launch day

Connector feature-requests per month; each dashed line is a connector going live.
The requests build up *before* a launch and drop right after.

In [ ]:
import matplotlib.dates as mdates
fr = df[(df.type == "feature_request") & (df.topic == "connectors")]
m = fr.groupby("month").size()
launches = {"Constant Contact": "2024-05-13", "Klaviyo": "2024-10-07",
            "Salesforce": "2025-03-03", "TikTok Ads": "2025-05-12", "Zoho CRM": "2025-09-15"}
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(m.index, m.values, marker="o", color=ACCENT, lw=2)
for name, d in launches.items():
    ts = pd.Timestamp(d); ax.axvline(ts, color=EVT_C, ls="--", alpha=0.7)
    ax.text(ts, ax.get_ylim()[1]*0.95, name, rotation=90, va="top", ha="right",
            fontsize=8, color=EVT_C)
ax.set_title("Connector feature-requests per month, with launch dates", fontweight="bold")
ax.set_ylabel("feature-requests"); ax.set_xlabel("")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=90, fontsize=7); plt.tight_layout(); plt.show()

## 3. Label distributions (the ML targets)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (col, order) in zip(axes.ravel(),
        [("priority", ["low","medium","high","critical"]), ("routing", None),
         ("topic", None), ("type", None)]):
    vc = df[col].value_counts()
    if order: vc = vc.reindex(order)
    (vc/len(df)*100).plot(kind="bar", ax=ax, color=ACCENT, width=0.8)
    ax.set_title(col, fontweight="bold"); ax.set_ylabel("% of tickets")
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
pd.crosstab(df.topic, df.type, margins=True, margins_name="Total")

## 4. Intake noise — the value of a triage classifier

`reported_category` is the customer's own dropdown pick; `aligned_category` is the
bucket a correctly self-tagging user should have chosen. The gap is what a model
recovers from the text.

In [ ]:
def aligned_category(topic, type_):
    """The picklist bucket a correctly self-tagging user would pick."""
    if type_ == "how_to": return "how_to_question"
    if type_ == "security": return "security_concern"
    if type_ in ("bug", "outage"): return "bug_something_broken"
    if topic == "billing" or type_ == "billing": return "account_billing"
    if topic == "connectors": return "connectors_integrations"
    if topic in ("dashboards", "reports", "northstar"): return "dashboards_reports"
    if topic == "attribution": return "attribution"
    if topic == "alerts": return "alerts"
    if topic == "users_workspace": return "users_access"
    return "other"

df["aligned"] = [aligned_category(t, ty) for t, ty in zip(df.topic, df.type)]
mismatch = (df.reported_category != df.aligned).mean() * 100
print(f"intake mismatch: {mismatch:.1f}% of tickets pick a category that disagrees "
      f"with the true labels")

**Takeaway:** ~a third of tickets arrive mis-categorized by the user. A model
that reads the text and predicts the true topic/type turns noisy intake into a
reliable routing signal — the concrete business value.

> Synthetic data (no real users, no PII). Scores on models trained here run high
> because text maps to labels cleanly by construction; expect lower on real tickets.